# 02 · Silver — Transacciones

Transforma `bronze.transacciones` en `silver.transacciones`: limpia, tipa, enriquece y **marca** los problemas de calidad sin descartar registros.

| | |
|---|---|
| **Origen** | `<catalog>.bronze.transacciones` |
| **Destino** | `<catalog>.silver.transacciones` y `<catalog>.silver.ref_feriados` |
| **Procesamiento** | Reconstrucción completa en cada corrida: idempotente por diseño (DT-012) |
| **Calidad** | Flags en `dq_flags`; solo se eliminan duplicados de `id_transaccion`, que siguen en Bronze |

| Regla | Criterio | Decisión |
|---|---|---|
| `DIA_INHABIL` | Fecha local en fin de semana o feriado → `fecha_operativa` = siguiente día hábil | DT-004 |
| Variantes MEP / Cable | `moneda = USD`: sufijo `D` → MEP, `C` → Cable; `simbolo_base` sin el sufijo | DT-005 |
| `PRECIO_FUERA_DE_RANGO` | Desvío mayor a 10x de la mediana del símbolo (mín. 5 operaciones, sin opciones) | DT-006 |
| `MONTO_EXTREMO` | Monto mayor al p99.9 de su tipo de instrumento | DT-006 |
| Tipo de instrumento | Reglas sobre símbolo y descripción; el resto se clasifica como `ACCION` | DT-011 |

**Prerrequisito:** `setup/04_silver.sql` y el notebook `01_bronze_ingesta`.

## Parámetros

In [0]:
dbutils.widgets.text("catalog", "workspace", "Catálogo")
CATALOG = dbutils.widgets.get("catalog")

In [0]:
import json
import os
import sys
from datetime import datetime, timezone

# La lógica vive en src/ para poder testearla sin cluster (ver tests/).
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..", "src")))

from pyspark.sql import functions as F

from iol_pipeline.config import PipelineConfig
from iol_pipeline.silver import (
    DQ_RULES,
    FERIADOS_AR,
    PRICE_DEVIATION_THRESHOLD,
    build_silver,
    holidays_df,
)

spark.conf.set("spark.sql.session.timeZone", "UTC")

cfg = PipelineConfig(catalog=CATALOG)
bronze_table = cfg.bronze_table("transacciones")
silver_table = cfg.silver_table("transacciones")
holidays_table = cfg.silver_table("ref_feriados")

print(f"Origen:   {bronze_table}")
print(f"Destino:  {silver_table}")
print(f"Feriados: {holidays_table}")

Origen:   workspace.bronze.transacciones
Destino:  workspace.silver.transacciones
Feriados: workspace.silver.ref_feriados


## Precondiciones

In [0]:
for table, script in [(bronze_table, "setup/02_bronze.sql"), (silver_table, "setup/04_silver.sql"), (holidays_table, "setup/04_silver.sql")]:
    assert spark.catalog.tableExists(table), f"No existe la tabla {table}. Ejecutar {script}"

bronze_rows = spark.table(bronze_table).count()
assert bronze_rows > 0, f"{bronze_table} está vacía. Ejecutar 01_bronze_ingesta"
print(f"Filas en bronze: {bronze_rows}")

Filas en bronze: 100000


## Calendario de feriados
La lista vive en `src/iol_pipeline/silver.py` (única fuente de verdad) y se publica como tabla para que sea consultable.

In [0]:
holidays_df(spark).writeTo(holidays_table).overwrite(F.lit(True))
display(spark.table(holidays_table).orderBy("fecha"))

fecha,descripcion
2026-01-01,Año Nuevo
2026-02-16,Carnaval
2026-02-17,Carnaval
2026-03-24,Día Nacional de la Memoria por la Verdad y la Justicia


## Transformación Bronze → Silver

In [0]:
processed_at = datetime.now(timezone.utc).replace(tzinfo=None)
result = build_silver(spark, bronze_table, silver_table, processed_at)
display(spark.createDataFrame([result.to_dict()]))

bronze_rows,duplicates_removed,message,rows_with_flags,rows_without_calendar,silver_rows,status,table
100000,0,"100000 filas en silver, 0 duplicados removidos, 10354 filas con flags de calidad",10354,0,100000,OK,workspace.silver.transacciones


## Reglas de calidad
Catálogo de reglas aplicadas (`DQ_RULES` en `src/iol_pipeline/silver.py`) y cuántas filas marcó cada una. Una fila puede tener más de un flag.

In [0]:
silver = spark.table(silver_table)
total = silver.count()

flag_counts = {
    r["flag"]: r["filas"]
    for r in silver.select(F.explode("dq_flags").alias("flag")).groupBy("flag").agg(F.count("*").alias("filas")).collect()
}
display(spark.createDataFrame(
    [
        (rule.name, rule.description, rule.severity, rule.condition,
         flag_counts.get(rule.name, 0), round(100 * flag_counts.get(rule.name, 0) / total, 3))
        for rule in DQ_RULES
    ],
    "regla STRING, descripcion STRING, severidad STRING, condicion STRING, filas_marcadas LONG, pct DOUBLE",
))

regla,descripcion,severidad,condicion,filas_marcadas,pct
DIA_INHABIL,Operación en fin de semana o feriado (hora local); se imputa al siguiente día hábil,WARN,NOT es_dia_habil,10255,10.255
PRECIO_FUERA_DE_RANGO,"Precio a más de 10x de la mediana de su símbolo y moneda (mínimo 5 operaciones, excluye opciones)",WARN,tipo_instrumento NOT IN ('OPCION') AND _n_obs >= 5 AND _desvio_precio > 10.0,8,0.008
MONTO_EXTREMO,Monto por encima del p99.9 de su tipo de instrumento,WARN,monto > _monto_limite,104,0.104


In [0]:
display(silver.groupBy("dq_status").agg(F.count("*").alias("filas")).withColumn("pct", F.round(100 * F.col("filas") / total, 2)).orderBy("dq_status"))

dq_status,filas,pct
OK,89646,89.65
WARN,10354,10.35


### Precios fuera de rango
Detalle por símbolo (sin datos de clientes). El umbral es un desvío mayor a `PRICE_DEVIATION_THRESHOLD` veces la mediana.

In [0]:
print(f"Umbral: {PRICE_DEVIATION_THRESHOLD}x la mediana del símbolo")
display(
    silver.where(F.array_contains("dq_flags", "PRECIO_FUERA_DE_RANGO"))
    .groupBy("simbolo", "moneda", "tipo_instrumento", "precio_referencia")
    .agg(F.count("*").alias("operaciones"), F.min("precio").alias("precio_min"), F.max("precio").alias("precio_max"))
    .orderBy("simbolo")
)

Umbral: 10.0x la mediana del símbolo


simbolo,moneda,tipo_instrumento,precio_referencia,operaciones,precio_min,precio_max
AL30,ARS,BONO,876.319200,1,0.010000,0.010000
DNC7D,USD,ON,1.071200,1,0.010000,0.010000
METR,ARS,ACCION,2252.824300,1,0.999700,0.999700
SATL,ARS,CEDEAR,5355.486300,1,1.004600,1.004600
T13F6,ARS,BONO,1.412000,1,0.010000,0.010000
T30A7,ARS,BONO,1.081400,1,0.010100,0.010100
T31Y7,ARS,BONO,1.021700,1,0.010000,0.010000
TGNO4,ARS,ACCION,4335.507650,1,0.999900,0.999900


### Días inhábiles
Operaciones en fin de semana o feriado y el día hábil al que se imputan.

In [0]:
display(
    silver.where(~F.col("es_dia_habil"))
    .groupBy("fecha_local", F.date_format("fecha_local", "EEEE").alias("dia_semana"), "fecha_operativa")
    .agg(F.count("*").alias("operaciones"))
    .orderBy("fecha_local")
)

fecha_local,dia_semana,fecha_operativa,operaciones
2026-01-03,Saturday,2026-01-05,1212
2026-01-10,Saturday,2026-01-12,1156
2026-01-17,Saturday,2026-01-19,962
2026-01-24,Saturday,2026-01-26,951
2026-01-31,Saturday,2026-02-02,1237
2026-02-07,Saturday,2026-02-09,1148
2026-02-14,Saturday,2026-02-18,854
2026-02-21,Saturday,2026-02-23,700
2026-02-28,Saturday,2026-03-02,1060
2026-03-07,Saturday,2026-03-09,975


## Enriquecimiento
### Tipo de instrumento

In [0]:
display(
    silver.groupBy("tipo_instrumento")
    .agg(F.count("*").alias("operaciones"), F.countDistinct("simbolo").alias("simbolos"))
    .withColumn("pct_operaciones", F.round(100 * F.col("operaciones") / total, 2))
    .orderBy(F.desc("operaciones"))
)

tipo_instrumento,operaciones,simbolos,pct_operaciones
CEDEAR,53962,595,53.96
BONO,18974,121,18.97
ACCION,18168,89,18.17
LETRA,4052,26,4.05
ON,2589,313,2.59
OPCION,2160,299,2.16
WARRANT,95,2,0.1


### Variantes de liquidación
`simbolos_con_base_operada`: variantes cuyo ticker base en ARS también aparece en el dataset. No es condición para clasificar la variante (DT-005), pero indica cuántas podrán usar la cotización del ticker base.

In [0]:
ars_symbols = silver.where("moneda = 'ARS'").select(F.col("simbolo").alias("simbolo_base")).distinct().withColumn("base_operada", F.lit(True))
display(
    silver.select("simbolo", "simbolo_base", "tipo_liquidacion").distinct()
    .join(ars_symbols, "simbolo_base", "left")
    .groupBy("tipo_liquidacion")
    .agg(F.count("*").alias("simbolos"), F.count("base_operada").alias("simbolos_con_base_operada"))
    .orderBy("tipo_liquidacion")
)

tipo_liquidacion,simbolos,simbolos_con_base_operada
ARS,974,974
CABLE,30,21
MEP,441,258


### Horario de mercado
Proporción de operaciones dentro del horario de BYMA (días hábiles, 11 a 17 hs).

In [0]:
display(
    silver.groupBy("es_dia_habil", "en_horario_mercado")
    .agg(F.count("*").alias("operaciones"))
    .withColumn("pct", F.round(100 * F.col("operaciones") / total, 2))
    .orderBy("es_dia_habil", "en_horario_mercado")
)

es_dia_habil,en_horario_mercado,operaciones,pct
false,false,10255,10.26
true,false,62169,62.17
true,true,27576,27.58


## Resultado
Se devuelve al orquestador como JSON.

In [0]:
summary = result.to_dict()
print(json.dumps(summary, indent=2, default=str))
dbutils.notebook.exit(json.dumps(summary, default=str))